In [1]:
pip install tensorflow.keras

Note: you may need to restart the kernel to use updated packages.


In [2]:
# load the trained model, scaler, pickle and onehot
import pickle
from tensorflow.keras.models import load_model
model = load_model('model.h5')

# load the encoders and scaler
with open('onehot_encoder_geo.pk1','rb') as file:
    label_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pk1','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pk1','rb') as file:
    scaler = pickle.load(file)

In [3]:
# Example input data:
input_data = {
    'CreditScore':80000,
    'Geography':'France',
    'Gender':'Female',
    'Age':40,
    'Tenure':4,
    'Balance':50000,
    'NumOfProducts':3,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [4]:
# onehot encode 'Geography':
import pandas as pd
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\Dell\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
# Here i/p data is in the key_value format, so we have to convert into dataframe
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,80000,France,Female,40,4,50000,3,1,1,50000


In [6]:
# Encode categorical variables
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,80000,France,0,40,4,50000,3,1,1,50000


In [7]:
# concatenation of onehot encoded
input_df = pd.concat([input_df.drop("Geography",axis=1), geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,80000,0,40,4,50000,3,1,1,50000,1.0,0.0,0.0


In [8]:
# scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[ 8.13439077e+02, -1.10158775e+00,  9.33786188e-02,
        -3.84185059e-01, -4.38198549e-01,  2.45913540e+00,
         6.49749829e-01,  9.74829420e-01, -8.37406840e-01,
         9.75804859e-01, -5.75619080e-01, -5.60226950e-01]])

In [9]:
# Prediction of churn
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step


array([[0.]], dtype=float32)

In [10]:
# Probability of the prediction
prediction_prob = prediction[0][0]
prediction_prob

0.0

In [11]:
if prediction_prob>0.5:
    print("The customer is likely to churn")
else:
    print("The customer is not likely to churn")

The customer is not likely to churn
